## Daten in CSV Umwandeln

In [ ]:
# Nur die ersten 20 Epochen speichern für Effizienz
import pandas as pd

# Einlesen der Datei
df = pd.read_csv(
    "../data/raw/data.txt",
    sep=r'\s+',  # Beliebige Leerzeichen als Trennzeichen
    names=["date", "time", "epoch", "moteid", "temperature", "humidity", "light", "voltage"],
    dtype={
        "epoch": "Int64",
        "moteid": "Int64",
        "temperature": "float",
        "humidity": "float",
        "light": "float",
        "voltage": "float"
    },
)

# --- FILTER: Nur die ersten 20 Epochen ---
df_filtered = df[df["epoch"] < 20].copy()

# Export in CSV
df_filtered.to_csv("../data/graph/data.csv", index=False)


## Daten für Baseline verschriftlichen

In [5]:
import csv
import json
from pathlib import Path

input_path = Path("../data/graph/data.csv")
output_path = Path("../data/base/data.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(input_path, "r", encoding="utf-8") as csv_file, \
        open(output_path, "w", encoding="utf-8") as jsonl_file:

    reader = csv.DictReader(csv_file, fieldnames=[
        "date", "time", "epoch", "moteid",
        "temperature", "humidity", "light", "voltage"
    ])

    for row in reader:
        # Werte extrahieren
        date = row["date"]
        time = row["time"]
        mote = row["moteid"]
        temp = row["temperature"]
        hum = row["humidity"]
        light = row["light"]
        volt = row["voltage"]
        epoch = row["epoch"]

        # Kurzsatz für BM25
        text = (
            f"Am {date} um {time} hat Sensor {mote} "
            f"{temp}°C Temperatur, {hum}% Luftfeuchtigkeit, "
            f"{light} Lux Licht und {volt}V Spannung gemessen."
        )

        # Haystack-kompatibles JSONL-Dokument
        doc = {
            "content": text,
            "meta": {
                "date": date,
                "time": time,
                "epoch": epoch,
                "moteid": mote,
                "temperature": temp,
                "humidity": hum,
                "light": light,
                "voltage": volt
            }
        }

        jsonl_file.write(json.dumps(doc, ensure_ascii=False) + "\n")
